In [1]:
# Install dependencies
%pip install ultralytics
%pip install opencv-python
%pip install simple_pid
%pip install numpy
%pip install pyserial
%pip install cv2-enumerate-cameras

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Configurations
import cv2
import math
import serial
import time
import numpy as np
from ultralytics import YOLO
from simple_pid import PID
from cv2_enumerate_cameras import enumerate_cameras

# Camera configuration
CAMERA_INDEX_1 = 702               # Index of the camera  to be used (custom value; check with enumerate_cameras)
CAMERA_INDEX_2 = 701               # Index of the camera to be used (custom value; check with enumerate_cameras)
FRAME_DELAY = 0.05                  # Delay between frames in seconds for processing

# Geometry setup
CAMERA_DISTANCE = 48               # Distance between two cameras (used for triangulation)
DISTANCE_UNIT = "cm"               # Unit of measurement for camera distance

# YOLO model setup
MODEL = YOLO('yolov10m.pt')         # Load YOLOv8 nano model for object detection
CLASS_NAMES = MODEL.names          # List of all class labels in the model
DESIRED_CLASSES = ["sports ball"]  # Classes we want to detect (only track sports balls)

# PID control parameters
PID_KP = 0.03                      # PID Proportional gain
PID_KI = 0.01                      # PID Integral gain
PID_KD = 0.03                      # PID Derivative gain
PID_OUTPUT_LIMITS = (-10, 10)      # Output limits for PID correction to smooth servo movement

# Servo configuration
SERVO_MIN = 0                      # Minimum angle a servo can rotate to
SERVO_MAX = 180                    # Maximum angle a servo can rotate to
SERVO_INIT = {                     # Initial servo positions (centered)
    'x1': 45, 
    'y1': 90, 
    'x2': 135, 
    'y2': 90
}

# Arduino serial communication
ARDUINO_PORT = 'COM3'              # Serial port where Arduino is connected
ARDUINO_BAUDRATE = 9600            # Communication speed between Python and Arduino

In [3]:
# CV2 and camera settings

# Initialize camera
def init_camera(index, verbose=True):
    cap = cv2.VideoCapture(index)
    if not cap.isOpened():
        raise RuntimeError(f"Error: Could not open webcam {index}.")
    if verbose:
        print(f"Camera {index} opened successfully.")
    return cap


# Read current frame
def read_frame(cap):
    ret, frame = cap.read()
    if not ret:
        raise RuntimeError("Failed to capture image from webcam.")
    return frame


# Calculate frame center
def calculate_center(frame):
    height, width = frame.shape[:2]
    x_center, y_center = width / 2, height / 2
    return x_center, y_center


# Check if class is in desired classes
def valid_class(box, class_names=CLASS_NAMES, desired_classes=DESIRED_CLASSES):
    return class_names[int(box.cls)] in desired_classes


# Draw bounding box
def draw_bounding_box(frame, box, class_names=CLASS_NAMES, verbose=True):
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 255), 3)
    
    cls = int(box.cls[0])
    confidence = math.ceil((box.conf[0] * 100)) / 100
    
    org = [x1, y1]
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1
    color = (255, 0, 0)
    thickness = 2
    cv2.putText(frame, class_names[cls], org, font, font_scale, color, thickness)
    
    if verbose:
        print(f"Class: {class_names[cls]}, Confidence: {confidence}")

In [4]:
# Calculations

# Calculate balloon coordinates
def triangulation(angles: dict, d=CAMERA_DISTANCE, verbose=True):
    alpha = np.radians(angles['x1'])
    beta = np.radians(angles['x2']-90)
    gamma = np.radians(angles['y1']-90)
    delta = np.radians(angles['y2']-90)
    
    if (np.sin(alpha + beta)) == 0:
        raise ValueError("Invalid angles for triangulation.")
    
    n = d * np.sin(beta) / np.sin(alpha + beta)
    m = d * np.sin(alpha) / np.sin(alpha + beta)

    x1 = n * np.cos(alpha)
    y1 = n * np.sin(alpha)
    z1 = n * np.tan(gamma)
    
    x2 = d - (m * np.cos(beta))
    y2 = m * np.sin(beta)
    z2 = m * np.tan(delta)

    x = (x1 + x2) / 2
    y = (y1 + y2) / 2
    z = (z1 + z2) / 2
    
    if verbose:
        print(f"Balloon position: ({x:.2f}, {y:.2f}, {z:.2f})")
    
    return np.array([x, y, z])


# Calculate distance to camera
def get_distance(camera, balloon_position, d=CAMERA_DISTANCE, verbose=True):
    if camera == 'C1':
        camera_position = np.array([0, 0, 0])
    elif camera == 'C2':
        camera_position = np.array([d, 0, 0])
    else:
        raise ValueError("Invalid camera ID. Use 'C1' or 'C2'.")
    
    distance = np.linalg.norm(balloon_position - camera_position)
    
    if verbose:
        print(f"{camera} Distance: {distance:.2f} {DISTANCE_UNIT}")
    
    return distance

In [5]:
# PID controller

# Initialize PID controller
def init_pid(x_center, y_center, verbose=True):
    pid_pan = PID(Kp=PID_KP, Ki=PID_KI, Kd=PID_KD, setpoint=x_center)
    pid_pan.output_limits = PID_OUTPUT_LIMITS
    pid_tilt = PID(Kp=PID_KP, Ki=PID_KI, Kd=PID_KD, setpoint=y_center)
    pid_tilt.output_limits = PID_OUTPUT_LIMITS
    if verbose:
        print("PID controller initialized.")
    return pid_pan, pid_tilt


# Calculate corrections using PID controller
def calculate_corrections(box, pid_pan, pid_tilt, verbose=True):
    x1, y1, x2, y2 = box.xyxy[0]
    x_box = float((x1 + x2) / 2)
    y_box = float((y1 + y2) / 2)
    pan, tilt = pid_pan(x_box), pid_tilt(y_box)
    if verbose:
        print(f"Box: ({x_box}, {y_box})")
        print(f"Pan: {pan}, Tilt: {tilt}")
    return pan, tilt


# Update servo angle
def update_angle(servo_angle, correction):
    if correction > 10:
        correction = 10
    elif correction < -10:
        correction = -10
    return max(SERVO_MIN, min(SERVO_MAX, math.ceil(servo_angle + correction)))

In [6]:
# Arduino

# Initialize serial connection to arduino
def init_arduino(*, port=ARDUINO_PORT, baudrate=ARDUINO_BAUDRATE, initial_angles=SERVO_INIT, verbose=True):
    try:
        arduino = serial.Serial(port, baudrate)
        time.sleep(2)
        if verbose:
            print(f"Connected to Arduino on {port} at {baudrate} baud.")
        move_servos(arduino, initial_angles)
        return arduino
    except serial.SerialException as e:
        raise RuntimeError(f"Could not connect to Arduino on {port}: {e}")


# Move servos
def move_servos(arduino, angles: dict, verbose=True):
    try:
        command = f"{angles['x1']},{angles['y1']},{angles['x2']},{angles['y2']}\n"
        arduino.write(command.encode())
        arduino.flush()
        if verbose:
            print(f"C1 → X: {angles['x1']:.1f}°, Y: {angles['y1']:.1f}°")
            print(f"C2 → X: {angles['x2']:.1f}°, Y: {angles['y2']:.1f}°")
            
    except serial.SerialException as e:
        print(f"Serial communication error: {e}")

In [7]:
# Print functions

# Print divider
def print_div(text):
    print("\n" + f" {text} ".center(100, '-') + "\n")


# Print available cameras
def print_cameras():
    for camera_info in enumerate_cameras():
        print(f'{camera_info.index}: {camera_info.name}')


# Print final information
def final_prints(frame_count, x1_center, y1_center, x2_center, y2_center, balloon_position, dist_c1, dist_c2, servo_angles):
    x, y, z = balloon_position
    print(f"Total frames: {frame_count}")
    print(f"Detected classes: {DESIRED_CLASSES}")
    print(f"Frame centers: ({x1_center}, {y1_center}), ({x2_center}, {y2_center})")
    print(f"FPS: {(1/FRAME_DELAY):.2f}")
    print(f"Final balloon position: ({x:.2f}, {y:.2f}, {z:.2f})")
    print(f"Final C1 distance: {dist_c1:.2f} {DISTANCE_UNIT}")
    print(f"Final C2 distance: {dist_c2:.2f} {DISTANCE_UNIT}")
    print(f"Servo angles: {servo_angles}\n")

In [19]:
# Main program
print_div("Available cameras")
print_cameras()

print_div("Initial settings")
frame_count = 0

SERVO_INIT = {'x1': 45, 'y1': 90, 'x2': 135, 'y2': 90}
servo_angles = SERVO_INIT
arduino = init_arduino(initial_angles=servo_angles)

cap1 = init_camera(CAMERA_INDEX_1)
cap2 = init_camera(CAMERA_INDEX_2)
frame1 = read_frame(cap1)
frame2 = read_frame(cap2)
x1_center, y1_center = calculate_center(frame1)
x2_center, y2_center = calculate_center(frame2)

pid_pan1, pid_tilt1 = init_pid(x1_center, y1_center)
pid_pan2, pid_tilt2 = init_pid(x2_center, y2_center)

print_div("Program started")
try:
    while True:
        x1_correction, y1_correction = 0, 0
        frame1 = read_frame(cap1)
        results1 = MODEL(frame1, stream=True)
        for r1 in results1:
            if not r1.boxes:
                continue
            for box1 in r1.boxes:
                if valid_class(box1):
                    draw_bounding_box(frame1, box1)
                    x1_correction, y1_correction = calculate_corrections(box1, pid_pan1, pid_tilt1)
        
        x2_correction, y2_correction = 0, 0
        frame2 = read_frame(cap2)
        results2 = MODEL(frame2, stream=True)
        for r2 in results2:
            if not r2.boxes:
                continue
            for box2 in r2.boxes:
                if valid_class(box2):
                    draw_bounding_box(frame2, box2)
                    x2_correction, y2_correction = calculate_corrections(box2, pid_pan2, pid_tilt2)
        
        print(f"Frame number: {frame_count}")
        balloon_position = triangulation(servo_angles)
        dist_c1 = get_distance('C1', balloon_position)
        dist_c2 = get_distance('C2', balloon_position)
        print(f"Servo angles: {servo_angles}")
        
        c1_change = x1_correction != 0 or y1_correction != 0
        c2_change = x2_correction != 0 or y2_correction != 0
        
        if c1_change:
            y1_correction *= -1 # type: ignore
            servo_angles['x1'] = update_angle(servo_angles['x1'], x1_correction)
            servo_angles['y1'] = update_angle(servo_angles['y1'], y1_correction)
        
        if c2_change:
            y2_correction *= -1 # type: ignore
            servo_angles['x2'] = update_angle(servo_angles['x2'], x2_correction)
            servo_angles['y2'] = update_angle(servo_angles['y2'], y2_correction)
        
        if c1_change or c2_change:
            move_servos(arduino, servo_angles)
        
        cv2.imshow("Image1", frame1)
        cv2.imshow("Image2", frame2)
        if cv2.waitKey(1) == ord('q'):
            break
        frame_count += 1
        time.sleep(FRAME_DELAY)
finally:
    print_div("Program finished")
    final_prints(frame_count, x1_center, y1_center, x2_center, y2_center, balloon_position, dist_c1, dist_c2, servo_angles)
    cap1.release()
    cap2.release()
    cv2.destroyAllWindows()
    arduino.close()


---------------------------------------- Available cameras -----------------------------------------

1400: Logi C270 HD WebCam
1401: Logi C270 HD WebCam
1402: HD User Facing
700: HD User Facing
701: Logi C270 HD WebCam
702: Logi C270 HD WebCam

----------------------------------------- Initial settings -----------------------------------------

Connected to Arduino on COM3 at 9600 baud.
C1 → X: 45.0°, Y: 90.0°
C2 → X: 135.0°, Y: 90.0°
Camera 702 opened successfully.
Camera 701 opened successfully.
PID controller initialized.
PID controller initialized.

----------------------------------------- Program started ------------------------------------------


0: 480x640 3 persons, 1 handbag, 403.9ms
Speed: 2.2ms preprocess, 403.9ms inference, 0.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 3 bottles, 476.5ms
Speed: 4.9ms preprocess, 476.5ms inference, 0.9ms postprocess per image at shape (1, 3, 480, 640)
Frame number: 0
Balloon position: (24.00, 24.00, 0.00)
C

In [12]:
enumerate_cameras()

[1400: Logi C270 HD WebCam (046D:0825),
 1401: HD User Facing (04F2:B64F),
 700: HD User Facing (04F2:B64F),
 701: Logi C270 HD WebCam (046D:0825)]